# 📄 Research Analysis: Cost-Sensitive Decision Optimization for Refund Systems

## Novel Contributions

This notebook presents four novel contributions to the field of cost-sensitive decision systems:

1. **Cost-Sensitive Custom Loss Training** — Per-instance sample weights derived from economic cost
2. **Optimal Decision Threshold Search** — Proving cost-optimal threshold ≠ 0.5
3. **Dynamic Cost Sensitivity Analysis** — How the optimal strategy changes under different cost regimes
4. **Pareto Front Analysis** — Multi-objective optimization of accuracy vs economic cost

---

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from src.config import Config
from src.data_generator import generate_dataset
from src.rule_engine import RuleEngine
from src.model import ModelPipeline, get_available_models
from src.metrics import EconomicMetrics, classification_metrics
from src.visualization import apply_style

# Novel modules
from src.cost_sensitive_model import CostSensitivePipeline, compute_sample_weights
from src.threshold_optimizer import (
    sweep_thresholds, find_optimal_thresholds, compare_thresholds_across_models
)
from src.sensitivity_analysis import (
    single_parameter_sweep, find_crossover_points, dual_parameter_heatmap, winner_summary
)
from src.pareto_analysis import (
    compute_strategy_objectives, find_pareto_front, pareto_summary, extended_pareto_with_thresholds
)

# Configuration
config = Config(random_seed=42, n_samples=1000)
apply_style()

print("✅ All modules loaded")
print(f"Available models: {get_available_models()}")

In [ ]:
# Generate data and train baseline models
data = generate_dataset(config=config)

# Standard ML pipeline
pipeline = ModelPipeline(config=config)
X_train, X_test, y_train, y_test = pipeline.prepare_data(data)
pipeline.train_all(tune_hyperparams=True)

# Rule-based predictions on test set
engine = RuleEngine()
test_data = data.loc[X_test.index].reset_index(drop=True)

rule_predictions = {
    f"Rule:{s}": engine.predict(test_data, strategy=s)
    for s in RuleEngine.STRATEGIES
}

ml_predictions = {
    f"ML:{name}": pipeline.predict(name)
    for name in pipeline.models
}

ml_probas = {
    name: pipeline.predict_proba(name)
    for name in pipeline.models
}

all_predictions = {**rule_predictions, **ml_predictions}

print(f"✅ Data generated: {data.shape}")
print(f"✅ {len(pipeline.models)} ML models trained")
print(f"✅ {len(all_predictions)} total strategies ready")

---
## Novel Contribution 1: Cost-Sensitive Custom Loss Training

### Hypothesis
Models trained with per-instance cost weights will achieve **lower economic cost** on the test set compared to standard accuracy-optimized models, even if their classification accuracy is slightly lower.

### Method
We derive sample weights from the economic cost model:
- Refundable orders (y=1): weight ∝ retention_value + 0.1 × order_amount
- Non-refundable orders (y=0): weight ∝ order_amount × fraud_multiplier

Weights are normalized (mean=1.0) and passed to `fit(sample_weight=...)`.

In [ ]:
# Train cost-sensitive models
cs_pipeline = CostSensitivePipeline(config=config)
cs_pipeline.prepare_data(data)
cs_pipeline.train_both()

# Compare standard vs cost-weighted
comparison = cs_pipeline.compare()
print("\n📊 Standard vs Cost-Weighted Training:")
print("=" * 65)
print(comparison.to_string(index=False))

In [ ]:
# Cost reduction analysis
reduction = cs_pipeline.get_cost_reduction()
print("\n💰 Cost Reduction from Cost-Weighted Training:")
print("=" * 65)
print(reduction.to_string(index=False))

In [ ]:
# Visualization: Standard vs Cost-Weighted
apply_style()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

models = comparison['Model'].unique()
x = np.arange(len(models))
w = 0.3

std_costs = comparison[comparison['Training_Type'] == 'Standard']['Economic_Cost'].values
cw_costs = comparison[comparison['Training_Type'] == 'Cost-Weighted']['Economic_Cost'].values
std_acc = comparison[comparison['Training_Type'] == 'Standard']['Accuracy'].values
cw_acc = comparison[comparison['Training_Type'] == 'Cost-Weighted']['Accuracy'].values

ax1.bar(x - w/2, std_costs, w, label='Standard', color='#f78166', edgecolor='#30363d')
ax1.bar(x + w/2, cw_costs, w, label='Cost-Weighted', color='#3fb950', edgecolor='#30363d')
ax1.set_xlabel('Model', fontweight='bold')
ax1.set_ylabel('Economic Cost (₹)', fontweight='bold')
ax1.set_title('Economic Cost: Standard vs Cost-Weighted', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=15, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2.bar(x - w/2, std_acc, w, label='Standard', color='#f78166', edgecolor='#30363d')
ax2.bar(x + w/2, cw_acc, w, label='Cost-Weighted', color='#3fb950', edgecolor='#30363d')
ax2.set_xlabel('Model', fontweight='bold')
ax2.set_ylabel('Accuracy', fontweight='bold')
ax2.set_title('Accuracy: Standard vs Cost-Weighted', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(models, rotation=15, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0, 1)

plt.suptitle('Novel Contribution 1: Cost-Sensitive Training Impact',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Novel Contribution 2: Optimal Decision Threshold Search

### Hypothesis
The cost-optimal decision threshold differs from the default 0.5, and using the optimal threshold yields significant cost savings.

### Method
For each ML model, sweep thresholds from 0.0 to 1.0 in 0.01 steps. At each threshold, compute both accuracy and economic cost.

In [ ]:
# Threshold sweep for all models
apply_style()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

model_names = list(ml_probas.keys())
colors_cost = '#f78166'
colors_acc = '#58a6ff'

for i, name in enumerate(model_names):
    if i >= 4:
        break
    ax = axes[i]
    proba = ml_probas[name]
    sweep = sweep_thresholds(y_test.values, proba, test_data, config, n_steps=100)
    optimal = find_optimal_thresholds(sweep)
    
    ax2 = ax.twinx()
    
    ax.plot(sweep['threshold'], sweep['economic_cost'], color=colors_cost, lw=2, label='Cost')
    ax2.plot(sweep['threshold'], sweep['accuracy'], color=colors_acc, lw=2, label='Accuracy')
    
    # Mark optimal thresholds
    ct = optimal['cost_optimal']['threshold']
    at = optimal['accuracy_optimal']['threshold']
    ax.axvline(ct, color=colors_cost, ls='--', alpha=0.7, label=f'Cost-optimal: {ct:.2f}')
    ax.axvline(at, color=colors_acc, ls='--', alpha=0.7, label=f'Acc-optimal: {at:.2f}')
    ax.axvline(0.5, color='white', ls=':', alpha=0.4, label='Default: 0.50')
    
    ax.set_xlabel('Threshold', fontweight='bold')
    ax.set_ylabel('Economic Cost (₹)', color=colors_cost, fontweight='bold')
    ax2.set_ylabel('Accuracy', color=colors_acc, fontweight='bold')
    ax.set_title(name, fontweight='bold')
    ax.legend(loc='upper left', fontsize=7)
    ax.grid(alpha=0.2)

for j in range(len(model_names), 4):
    axes[j].set_visible(False)

plt.suptitle('Novel Contribution 2: Threshold vs Cost/Accuracy Curves',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare optimal thresholds across models
threshold_comparison = compare_thresholds_across_models(
    y_test.values, ml_probas, test_data, config
)

print("\n📊 Optimal Threshold Comparison:")
print("=" * 90)
print(threshold_comparison.to_string(index=False, float_format='%.3f'))

print("\n🔑 Key Finding:")
avg_gap = threshold_comparison['Threshold_Gap'].mean()
avg_savings = threshold_comparison['Cost_Savings_vs_Default'].mean()
print(f"   Average gap between cost-optimal and accuracy-optimal threshold: {avg_gap:.3f}")
print(f"   Average cost savings vs default 0.5 threshold: ₹{avg_savings:,.0f}")

---
## Novel Contribution 3: Dynamic Cost Sensitivity Analysis

### Hypothesis
The optimal strategy is not fixed — it depends on the cost environment. As retention costs or fraud penalties change, the best strategy switches.

### Method
1. Sweep `retention_value` from ₹100 to ₹2000
2. Sweep `fraud_penalty_multiplier` from 1.0× to 5.0×
3. Build a 2D heatmap showing which strategy wins at each (retention, fraud) pair

In [ ]:
# Single parameter sweep: retention_value
ret_range = np.linspace(100, 2000, 30)

sweep_ret = single_parameter_sweep(
    test_data, all_predictions,
    'retention_value', ret_range, config
)

apply_style()
fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#f0e68c', '#ff7b72', '#79c0ff']

for i, strategy in enumerate(all_predictions.keys()):
    subset = sweep_ret[sweep_ret['strategy'] == strategy]
    color = colors[i % len(colors)]
    ax.plot(subset['parameter_value'], subset['total_cost'],
            label=strategy, color=color, lw=2)

ax.set_xlabel('Retention Value (₹)', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Economic Cost (₹)', fontsize=12, fontweight='bold')
ax.set_title('Strategy Costs vs Retention Value', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Single parameter sweep: fraud_penalty_multiplier
fraud_range = np.linspace(1.0, 5.0, 30)

sweep_fraud = single_parameter_sweep(
    test_data, all_predictions,
    'fraud_penalty_multiplier', fraud_range, config
)

apply_style()
fig, ax = plt.subplots(figsize=(12, 7))

for i, strategy in enumerate(all_predictions.keys()):
    subset = sweep_fraud[sweep_fraud['strategy'] == strategy]
    color = colors[i % len(colors)]
    ax.plot(subset['parameter_value'], subset['total_cost'],
            label=strategy, color=color, lw=2)

ax.set_xlabel('Fraud Penalty Multiplier', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Economic Cost (₹)', fontsize=12, fontweight='bold')
ax.set_title('Strategy Costs vs Fraud Penalty Severity', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 2D Heatmap: which strategy wins?
ret_grid = np.linspace(100, 2000, 25)
fraud_grid = np.linspace(1.0, 5.0, 25)

heatmap_result = dual_parameter_heatmap(
    test_data, all_predictions,
    ret_grid, fraud_grid, config
)

# Winner summary
win_summary = winner_summary(heatmap_result)
print("\n📊 Strategy Win Percentages Across Cost Parameter Space:")
print("=" * 50)
print(win_summary.to_string(index=False))

In [ ]:
# Visualize the best-strategy heatmap
apply_style()
fig, ax = plt.subplots(figsize=(12, 9))

strategy_names = heatmap_result['strategy_names']
n_strategies = len(strategy_names)

# Create colormap
cmap = plt.cm.get_cmap('tab10', n_strategies)

im = ax.imshow(
    heatmap_result['best_strategy_grid'],
    aspect='auto', origin='lower',
    extent=[fraud_grid[0], fraud_grid[-1], ret_grid[0], ret_grid[-1]],
    cmap=cmap, vmin=-0.5, vmax=n_strategies - 0.5,
)

cbar = plt.colorbar(im, ax=ax, ticks=range(n_strategies))
cbar.ax.set_yticklabels([s[:20] for s in strategy_names], fontsize=8)
cbar.set_label('Winning Strategy', fontweight='bold')

ax.set_xlabel('Fraud Penalty Multiplier', fontsize=12, fontweight='bold')
ax.set_ylabel('Retention Value (₹)', fontsize=12, fontweight='bold')
ax.set_title('Optimal Strategy Map Across Cost Parameter Space',
             fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

print("\n🔑 Key Finding: The optimal strategy is NOT fixed — it depends on the cost environment.")

---
## Novel Contribution 4: Pareto Front Analysis

### Hypothesis
When treating accuracy and cost as competing objectives, only a subset of strategies are Pareto-optimal (no other strategy beats them on BOTH metrics). The remaining strategies are dominated and should be discarded.

### Method
1. Plot all strategies in accuracy × cost space
2. Identify Pareto-optimal strategies using dominance sorting
3. Extend analysis with multiple thresholds per model

In [ ]:
# Basic Pareto analysis
objectives = compute_strategy_objectives(
    y_test.values, all_predictions, test_data, config
)

pareto = find_pareto_front(objectives)

print(pareto_summary(pareto))
print("\n" + "=" * 60)
print("\nFull Results:")
print(pareto[['Strategy', 'Accuracy', 'Economic_Cost', 'Is_Pareto_Optimal']]
      .sort_values('Economic_Cost').to_string(index=False))

In [ ]:
# Pareto front visualization
apply_style()
fig, ax = plt.subplots(figsize=(12, 8))

optimal = pareto[pareto['Is_Pareto_Optimal']]
dominated = pareto[~pareto['Is_Pareto_Optimal']]

ax.scatter(dominated['Accuracy'], dominated['Economic_Cost'],
           s=120, color='#8b949e', edgecolors='white', linewidth=1,
           label='Dominated', alpha=0.6, zorder=3)

ax.scatter(optimal['Accuracy'], optimal['Economic_Cost'],
           s=180, color='#3fb950', edgecolors='white', linewidth=2,
           label='Pareto-Optimal', zorder=5, marker='*')

# Draw Pareto front line
front = optimal.sort_values('Accuracy')
ax.plot(front['Accuracy'], front['Economic_Cost'],
        '--', color='#3fb950', alpha=0.5, lw=1.5)

# Label all points
for _, row in pareto.iterrows():
    short = row['Strategy'].replace('Rule:', 'R:').replace('ML:', 'M:')
    ax.annotate(short, (row['Accuracy'], row['Economic_Cost']),
                textcoords='offset points', xytext=(8, 5),
                fontsize=7, color='#c9d1d9')

ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_ylabel('Economic Cost (₹)', fontsize=12, fontweight='bold')
ax.set_title('Pareto Front: Accuracy vs Economic Cost',
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Extended Pareto: include multiple thresholds per model
ext_pareto = extended_pareto_with_thresholds(
    y_test.values, ml_probas, rule_predictions,
    test_data, config, threshold_steps=20
)

optimal_ext = ext_pareto[ext_pareto['Is_Pareto_Optimal']]
dominated_ext = ext_pareto[~ext_pareto['Is_Pareto_Optimal']]

print(f"\n📊 Extended Pareto Analysis ({len(ext_pareto)} strategies):")
print(f"   Pareto-optimal: {len(optimal_ext)}")
print(f"   Dominated:      {len(dominated_ext)}")
print(f"\n🏆 Pareto-Optimal Strategies:")
print(optimal_ext[['Strategy', 'Accuracy', 'Economic_Cost']]
      .sort_values('Accuracy').to_string(index=False))

In [ ]:
# Extended Pareto front visualization
apply_style()
fig, ax = plt.subplots(figsize=(14, 9))

# Color by type: rule vs ML
for _, row in dominated_ext.iterrows():
    if 'Rule' in row['Strategy'] or '@' not in row['Strategy']:
        c = '#f78166'
    else:
        c = '#58a6ff'
    ax.scatter(row['Accuracy'], row['Economic_Cost'],
               s=40, color=c, alpha=0.3, zorder=2)

for _, row in optimal_ext.iterrows():
    ax.scatter(row['Accuracy'], row['Economic_Cost'],
               s=150, color='#3fb950', edgecolors='white', linewidth=2,
               zorder=5, marker='*')
    short = row['Strategy'].replace('Rule:', 'R:').replace('ML:', 'M:')
    ax.annotate(short, (row['Accuracy'], row['Economic_Cost']),
                textcoords='offset points', xytext=(8, 5),
                fontsize=7, color='#c9d1d9')

# Draw Pareto front
front = optimal_ext.sort_values('Accuracy')
ax.plot(front['Accuracy'], front['Economic_Cost'],
        '--', color='#3fb950', alpha=0.6, lw=1.5)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f78166', markersize=8, label='Rule-based (dominated)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#58a6ff', markersize=8, label='ML@threshold (dominated)'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor='#3fb950', markersize=14, label='Pareto-optimal'),
]
ax.legend(handles=legend_elements, fontsize=10, framealpha=0.9)

ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_ylabel('Economic Cost (₹)', fontsize=12, fontweight='bold')
ax.set_title('Extended Pareto Front (All Strategies + ML Thresholds)',
             fontsize=14, fontweight='bold', pad=15)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary of Findings

### Contribution 1: Cost-Sensitive Training
Per-instance cost-weighted training shifts model behavior toward economic optimization. The trained models may sacrifice marginal accuracy to avoid expensive misclassifications.

### Contribution 2: Threshold Optimization
The cost-optimal decision threshold consistently differs from the statistical default of 0.5. Using the optimal threshold yields measurable cost savings without requiring model retraining.

### Contribution 3: Sensitivity Analysis
The optimal strategy is environment-dependent. Under high retention costs, lenient strategies dominate. Under high fraud penalties, conservative approaches win. No single strategy is universally optimal.

### Contribution 4: Pareto Analysis
Multi-objective analysis reveals that most strategies are dominated. The Pareto front provides decision-makers with a clear set of non-dominated alternatives, showing the exact tradeoff between accuracy and cost.

### Implications for Practice
1. **Do not use accuracy as the sole metric** for refund decision systems
2. **Tune decision thresholds** based on business cost structure, not statistical convention
3. **Re-evaluate strategy selection** when cost parameters change
4. **Use Pareto analysis** to present stakeholders with clear tradeoff options